#### 오답 구조와 프롬프트 설계가 성능에 영향을 미치는지 확인

In [1]:
import json
import os
from collections import Counter,defaultdict
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
LABELS = ['DEF', 'RIGHT', 'PROC', 'ORG', 'CRIT', 'ETC']
LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}

sample_data = [
    {'id': 'D01', 'category': 'DEF', 'text': '이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
    {'id': 'D02', 'category': 'DEF', 'text': '이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.'},
    {'id': 'D03', 'category': 'DEF', 'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.'},
    {'id': 'R01', 'category': 'RIGHT', 'text': '모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.'},
    {'id': 'R02', 'category': 'RIGHT', 'text': '사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
    {'id': 'P01', 'category': 'PROC', 'text': '이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.'},
    {'id': 'P02', 'category': 'PROC', 'text': '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
    {'id': 'O01', 'category': 'ORG', 'text': '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.'},
    {'id': 'O02', 'category': 'ORG', 'text': '위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'},
    {'id': 'C01', 'category': 'CRIT', 'text': '후보자는 선거일 현재 25세 이상인 국민이어야 한다.'},
    {'id': 'C02', 'category': 'CRIT', 'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.'},
    {'id': 'E01', 'category': 'ETC', 'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.'},
    {'id': 'E02', 'category': 'ETC', 'text': '이 법 시행 당시 종전의 규정에 따라 한 처분은 이 법에 따른 처분으로 본다.'},
]

df_all = pd.DataFrame(sample_data)
df = df_all.copy()

print(f'평가 데이터 로드 완료: {len(df)}건')
df.head()

평가 데이터 로드 완료: 13건


,id,category,text
0,D01,DEF,이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
1,D02,DEF,이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.
2,D03,DEF,"공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다."
3,R01,RIGHT,"모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 ..."
4,R02,RIGHT,사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.


##### 재현 가능한 평가 데이터
- 동일한 데이터와 평가기준

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
SYSTEM_PROMPT = (
    '너는 한국 법률 조항 분류기다. 반드시 다음 6개 코드 중 하나만 선택하라: ' + ', '.join(LABELS) + '. '
    '응답은 반드시 JSON 한 줄로만 반환한다. 형식: {"label":"DEF","reason":"..."}'
)
SYSTEM_PROMPT

'너는 한국 법률 조항 분류기다. 반드시 다음 6개 코드 중 하나만 선택하라: DEF, RIGHT, PROC, ORG, CRIT, ETC. 응답은 반드시 JSON 한 줄로만 반환한다. 형식: {"label":"DEF","reason":"..."}'

In [4]:
LABEL_GUIDE = '\n'.join([f'- {label}: {description}' for label, description in LABEL_DESC.items()])
def build_prompt(text, mode='zero', examples=None, rules=None, label_only=False):
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
    ]
    if rules:
        base += ['', '구분 규칙:']
        base += [f'- {rule}' for rule in rules]
    if mode == 'one' and examples:
        example = examples[0]
        base += [
            '',
            '예시 1개:',
            f"문장: {example['text']}",
            '정답(JSON): ' + json.dumps(example['output'], ensure_ascii=False),
        ]
    elif mode == 'few' and examples:
        base += ['', f'예시 {len(examples)}개:']
        for index, example in enumerate(examples, 1):
            base += [
                f'예시 {index}:',
                f"문장: {example['text']}",
                '정답(JSON): ' + json.dumps(example['output'], ensure_ascii=False),
            ]
    base += ['', f'분류할 문장: {text}']
    if label_only:
        base += ['JSON은 label만 포함하라.', '{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC"}']
    else:
        base += ['JSON으로만 응답하라.', '{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC","reason":"짧은 근거"}']
    return '\n'.join(base)

In [5]:
print(build_prompt(df['text'][0]))

다음 법률 문장을 6개 코드 중 하나로 분류하라.
라벨 설명:
- DEF: 정의/목적/적용범위 조항
- RIGHT: 권리/의무/금지/책임 조항
- PROC: 신청/심사/조사/불복/처벌 절차 조항
- ORG: 기관/위원회/법원 등 조직의 설치/구성/권한 조항
- CRIT: 자격/요건/기준/기간/수치 조건 조항
- ETC: 시행일/경과조치/위임 등 기타 조항

분류할 문장: 이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
JSON으로만 응답하라.
{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC","reason":"짧은 근거"}


In [6]:
def load_hf_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype='auto',device_map='auto'
    )
    return model, tokenizer

def extract_json(text):
    start = text.find('{')
    end = text.find('}')
    if start == -1 or end == -1 or end<=start:
        return None
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return None
def predict(model, tokenizer, prompt, max_new_tokens=128, do_sample=False, temperature=None, top_p=None):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors='pt').to(model.device)
    generate_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': do_sample,
        'pad_token_id': tokenizer.eos_token_id,
    }
    if temperature is not None:
        generate_kwargs['temperature'] = temperature
    if top_p is not None:
        generate_kwargs['top_p'] = top_p
    generated_ids = model.generate(**model_inputs, **generate_kwargs)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed = extract_json(raw_response)
    label = parsed.get('label') if parsed else 'PARSE_FAIL'
    reason = parsed.get('reason') if parsed else raw_response[:160]
    return {
        'raw_response': raw_response,
        'label': label,
        'reason': reason,
    }


def normalize_prediction(label):
    if isinstance(label, str) and label in LABELS:
        return label
    return 'PARSE_FAIL'


print('프롬프트/추론 함수 정의 완료')    

프롬프트/추론 함수 정의 완료


In [7]:
BASE_ZERO_FEW_EXAMPLES = [
    {'text':'신청인은 처분 통지를 받은 날부터 30일 이내에 이의신철을 할 수 있다', 'output': {'label' :'PROC','reason':'불복 절차를 규정한다' }}
]
def run_eval(model, tokenizer, mode='zero', label_only=False, rules=None,predict_kwargs=None):
    rows = []
    for sample in df.to_dict('records'):
        prompt = build_prompt(
            sample['text'],
            mode=mode,
            examples=BASE_ZERO_FEW_EXAMPLES if mode in ('one', 'few') else None,
            rules=rules,
            label_only=label_only,
        )
        predict_args = predict_kwargs or {}
        result = predict(model, tokenizer, prompt, max_new_tokens=128, **predict_args)
        rows.append({
            'id': sample['id'],
            'true_label': sample['category'],
            'predicted_label': normalize_prediction(result['label']),
            'reason': result['reason'],
            'raw_response': result['raw_response'],
            'correct': normalize_prediction(result['label']) == sample['category'],
            'text': sample['text'],
        })
    return pd.DataFrame(rows)

base_model, base_tokenizer =  load_hf_model(BASE_MODEL)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [8]:
base_pred_df = run_eval(base_model,base_tokenizer)
base_pred_df

,id,true_label,predicted_label,reason,raw_response,correct,text
0,D01,DEF,DEF,국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 하는 법입니다.,"{""label"":""DEF"", ""reason"":""국민의 기본적 인권을 보호하고 자유와...",True,이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
1,D02,DEF,DEF,정의/목적/적용범위,"{""label"":""DEF"", ""reason"":""정의/목적/적용범위""}",True,이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.
2,D03,DEF,DEF,"국가기관, 지방자치단체 및 법령에 따라 설치된 기관","{""label"":""DEF"", ""reason"":""국가기관, 지방자치단체 및 법령에 따...",True,"공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다."
3,R01,RIGHT,PARSE_FAIL,문장 내용은 법적 제도를 적용하는 것에 대한 명시적인 조항입니다.,"{""label"":""DEF|RIGHT|PROC|ORG|CRIT|ETC"",""reason...",False,"모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 ..."
4,R02,RIGHT,PARSE_FAIL,작업자와 이용자가 개인정보를 안전하게 관리해야 합니다.,"{""label"":""DEF|RIGHT|PROC|ORG|CRIT|ETC"",""reason...",False,사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.
5,P01,PROC,DEF,정의/목적/적용범위,"{""label"":""DEF"", ""reason"":""정의/목적/적용범위""}",False,이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.
6,P02,PROC,DEF,30일 이내에 이의신청을 할 수 있다,"{""label"":""DEF"", ""reason"":""30일 이내에 이의신청을 할 수 있다""}",False,신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.
7,O01,ORG,DEF,국무총리는 분쟁 조정을 위한 조정위원회를 설립해야 합니다.,"{""label"":""DEF"", ""reason"":""국무총리는 분쟁 조정을 위한 조정위원...",False,분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.
8,O02,ORG,DEF,정의/목적/적용범위,"{""label"":""DEF"", ""reason"":""정의/목적/적용범위""}",False,위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.
9,C01,CRIT,DEF,후보자가 선거일 현재 25세 이상인 국민이야,"{""label"":""DEF"", ""reason"":""후보자가 선거일 현재 25세 이상인 ...",False,후보자는 선거일 현재 25세 이상인 국민이어야 한다.


In [10]:
def metric_summary(pred_df):
    accuracy = float(pred_df['correct'].mean())
    labels = sorted(df['category'].unique())
    precisions = []
    recalls = []
    f1s = []
    for label in labels:
        tp = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] == label)).sum())
        fp = int(((pred_df['true_label'] != label) & (pred_df['predicted_label'] == label)).sum())
        fn = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] != label)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return {
        'accuracy': accuracy,
        'macro_precision': float(np.mean(precisions)),
        'macro_recall': float(np.mean(recalls)),
        'macro_f1': float(np.mean(f1s)),
    }

metric_summary(base_pred_df)

{'accuracy': 0.23076923076923078,
 'macro_precision': 0.049999999999999996,
 'macro_recall': 0.16666666666666666,
 'macro_f1': 0.07692307692307691}

##### 라벨별 혼동 행렬

In [11]:
label_order = LABELS
label_to_index =  {label:index for index, label in enumerate(label_order)}

def confusion_matrix_from_df(pred_df):
    matrix = np.zeros((len(label_order), len(label_order)), dtype=int)
    for _, row in pred_df.iterrows():
        true_label = row['true_label']
        pred_label = row['predicted_label']
        if true_label in label_to_index and pred_label in label_to_index:
            matrix[label_to_index[true_label], label_to_index[pred_label]] += 1
    return matrix


def per_label_metrics(pred_df):
    rows = []
    for label in label_order:
        tp = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] == label)).sum())
        fp = int(((pred_df['true_label'] != label) & (pred_df['predicted_label'] == label)).sum())
        fn = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] != label)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        rows.append({
            'label': label,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': int((pred_df['true_label'] == label).sum()),
        })
    return pd.DataFrame(rows)

confusion = confusion_matrix_from_df(base_pred_df)
label_metrics = per_label_metrics(base_pred_df)

In [12]:
print('혼동행렬')
pd.DataFrame(confusion,index = label_order,columns=label_order)

혼동행렬


,DEF,RIGHT,PROC,ORG,CRIT,ETC
DEF,3,0,0,0,0,0
RIGHT,0,0,0,0,0,0
PROC,2,0,0,0,0,0
ORG,2,0,0,0,0,0
CRIT,1,0,0,0,0,0
ETC,2,0,0,0,0,0


In [13]:
print('라벨별 지표')
print(label_metrics)

라벨별 지표
   label  precision  recall        f1  support
0    DEF        0.3     1.0  0.461538        3
1  RIGHT        0.0     0.0  0.000000        2
2   PROC        0.0     0.0  0.000000        2
3    ORG        0.0     0.0  0.000000        2
4   CRIT        0.0     0.0  0.000000        2
5    ETC        0.0     0.0  0.000000        2


#### 오답 유형 묶기
- 라벨형태로 묶어 원인을 분석